**Carregar e Preparar os Dados**

In [10]:
#Importação de Arquivos
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import joblib

In [4]:
#Criação DF
df = pd.read_csv("flights_delays_120.csv")

In [5]:
#Separação dos dados
df_features = df.drop("delayed", axis=1)
df_alvo = df['delayed']

In [6]:
#Tratamento dos dados
df_features_tratada = pd.get_dummies(df_features, dtype=int)

In [9]:
#Separação de dados para treino
x_treino, x_teste, y_treino, y_teste = train_test_split(
    df_features_tratada,
    df_alvo,
    test_size=0.3,
    random_state=42,
    stratify=df_alvo
)

**Treinar e Salvar o Modelo Localmente**

In [11]:
#Treinamento do modelo
modelo = XGBClassifier(random_state=42)
modelo.fit(x_treino, y_treino)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [12]:
#Salvamento dos dados
joblib.dump(modelo, "modelo_treinado.joblib")
print("Salvo com Sucesso.")

Salvo com Sucesso.


**Predições em Tempo Real - Simulação**

In [13]:
#Análise e simulação do modelo salvo.
modelo_salvo = joblib.load("modelo_treinado.joblib")

In [14]:
#Iniciando Simulação(APENAS 1 VOO)
voo_simulado = x_teste.iloc[0].values.reshape(1, -1)

In [16]:
#Obter dados de Previsão e Atraso
classe_prevista = modelo_salvo.predict(voo_simulado)[0]
probabilidade = modelo_salvo.predict_proba(voo_simulado)[0][1]

print("--- Relatório em Tempo Real ---")
print(f"Classe Prevista: {'Voo Atrasado' if classe_prevista == 1 else 'Voo no Horário Correto'}")
print(f"Probabilidade de Atraso: {probabilidade * 100:.2f}%")

--- Relatório em Tempo Real ---
Classe Prevista: Voo no Horário Correto
Probabilidade de Atraso: 9.62%


**Inferência em Lote - Batch Transform**

In [17]:
#Várias Previsões e Probabilidades
ls_previsoes = modelo_salvo.predict(x_teste)
ls_probabilidades = modelo_salvo.predict_proba(x_teste)[:, 1]

In [18]:
#Criação de DF
df_previsoes = x_teste.copy()
df_previsoes['Previsao_Atraso'] = ls_previsoes
df_previsoes['Probabilidade_Atraso'] = ls_probabilidades

In [19]:
#Listagem das novas colunas
df_previsoes.head()

,departure_hour,day_of_week,airline_AirOne,airline_FlyFast,airline_JetCloud,airline_SkyWings,airline_TravelAir,origin_BSB,origin_CNF,origin_GIG,...,destination_FOR,destination_REC,destination_SSA,weather_Clear,weather_Fog,weather_Rain,weather_Storm,weather_Wind,Previsao_Atraso,Probabilidade_Atraso
41,21,4,1,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0.096192
75,4,7,0,1,0,0,0,1,0,0,...,0,1,0,0,0,0,0,1,0,0.035355
98,7,2,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,1,0.847099
55,4,7,1,0,0,0,0,0,0,1,...,0,0,0,0,0,1,0,0,1,0.948960
76,12,2,1,0,0,0,0,1,0,0,...,1,0,0,0,0,0,0,1,0,0.012112


In [21]:
#Salvar dados em CSV
df_previsoes.to_csv("previsoes_lote.csv", index=False)

print("-- Inferência Completa --\nResultados Salvos com Sucesso!")

-- Inferência Completa --
Resultados Salvos com Sucesso!
